# 04 - NMRF Chain: Method Selection to SES

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [IMA package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: scenario P&L cubes with scenario metadata, risk-factor definitions, RFET real-price observations and evidence, stress-history series, PLA/backtesting observations, and NMRF valuation artifacts. The package dataset contract defines the committed fixture files, manifest lineage, and Arrow handoff: [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md). The staged workflow is described in the [IMA package journey](../docs/PACKAGE_JOURNEY.md).

This notebook reviews the Non-Modellable Risk Factor chain for the committed `capital_run_v1` fixture: RFET classification input, method-selection evidence, valuation spec handoff, returned-artifact reconciliation, and SES aggregation.

Classifications are re-derived from RFET evidence here and are expected to reconcile to the fixture golden classifications; capital notebooks 03 and 06 treat those classifications as an upstream input.

Regulatory anchors: Basel MAR33 NMRF stressed expected shortfall concepts; U.S. NPR 2.0 proposed-rule parameters for Type A / Type B NMRF routing, SES extraction, and conservative Type B aggregation cited below; EU CRR Article 325bk as comparison context.

Prototype caution: direct, stepwise, and full-revaluation pricing remains upstream. The fixture contains synthetic upstream valuation artifacts, and this notebook validates/uses them for prototype evidence only. Outputs are not final regulatory capital.


## Public API happy path

This notebook routes NMRFs, selects stress methods, builds valuation specs, and computes SES with public IMA NMRF APIs.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from examples.capital_run_fixture import (
    as_of_date_from_fixture,
    classifications_from_fixture,
    load_capital_run_v1_fixture,
    nmrf_artifacts_from_fixture,
    nmrf_direct_shocks_from_fixture,
    nmrf_full_revaluations_from_fixture,
    nmrf_method_evidence_from_fixture,
    policy_from_fixture,
)
from frtb_ima import (
    build_nmrf_valuation_run_request,
    build_nmrf_valuation_specs,
    calculate_nmrf_capital_from_valuation_run,
    complete_nmrf_valuation_run,
    route_nmrf_classifications_for_capital,
    select_nmrf_methods,
    select_stress_periods_for_policy,
    selection_input_from_method_evidence,
    stress_period_specs_for_nmrf,
    validate_selected_stress_periods,
)

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

fixture = load_capital_run_v1_fixture()
policy = policy_from_fixture(fixture)
as_of_date = as_of_date_from_fixture(fixture)
run_id = str(fixture.params["run_id"])
desk_id = str(fixture.params["desk_id"])

classifications = classifications_from_fixture(fixture, policy)
routing = route_nmrf_classifications_for_capital(classifications, policy)
nmrf_names = tuple(routing.ses_risk_factors)
risk_factor_by_name = {risk_factor.name: risk_factor for risk_factor in fixture.risk_factors}

stress_selection = select_stress_periods_for_policy(
    fixture.stress_histories,
    policy,
    as_of_date=as_of_date,
    run_id=run_id,
    desk_id=desk_id,
)
validate_selected_stress_periods(
    stress_selection,
    tuple(risk_factor_by_name[name].risk_class for name in nmrf_names),
)
stress_periods = stress_period_specs_for_nmrf(stress_selection)

method_evidence = nmrf_method_evidence_from_fixture(fixture)
selection_inputs = tuple(
    selection_input_from_method_evidence(
        method_evidence[name],
        classifications[name],
        risk_factor_by_name[name].liquidity_horizon,
    )
    for name in nmrf_names
)
decisions = select_nmrf_methods(selection_inputs, policy)
decision_by_name = {decision.risk_factor_name: decision for decision in decisions}
input_by_name = {item.risk_factor_name: item for item in selection_inputs}

specs = build_nmrf_valuation_specs(
    tuple(decision.to_valuation_instruction() for decision in decisions),
    {risk_factor.name: risk_factor.risk_class for risk_factor in fixture.risk_factors},
    stress_periods,
    policy,
    direct_shocks=nmrf_direct_shocks_from_fixture(fixture),
    full_revaluations=nmrf_full_revaluations_from_fixture(fixture),
    source="fixture NMRF valuation spec builder",
)
artifacts = nmrf_artifacts_from_fixture(fixture, specs)
artifact_by_name = {artifact.risk_factor_name: artifact for artifact in artifacts}
valuation_request = build_nmrf_valuation_run_request(
    specs,
    policy,
    run_id=run_id,
    desk_id=desk_id,
    as_of_date=as_of_date,
)
valuation_run = complete_nmrf_valuation_run(valuation_request, artifacts)
nmrf_capital = calculate_nmrf_capital_from_valuation_run(classifications, valuation_run, policy)
ses_results_by_name = {
    result.risk_factor_name: result
    for result in (*nmrf_capital.type_a_results, *nmrf_capital.type_b_results)
}


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


def golden_check(name: str, actual: float) -> tuple[float, float, str]:
    spec = fixture.expected_outputs["scalars"][name]
    expected = float(spec["value"])
    tolerance = float(spec.get("atol", 1e-9)) + float(spec.get("rtol", 1e-9)) * abs(expected)
    diff = actual - expected
    return expected, diff, "PASS" if abs(diff) <= tolerance else "CHECK"


def tail_summary(values: np.ndarray, alpha: float) -> tuple[int, float, float]:
    sorted_desc = np.sort(np.asarray(values, dtype=float))[::-1]
    tail_count = max(1, int(np.ceil(sorted_desc.size * (1.0 - alpha))))
    tail_cutoff = float(sorted_desc[tail_count - 1])
    tail_mean = float(np.mean(sorted_desc[:tail_count]))
    return tail_count, tail_cutoff, tail_mean


display(
    Markdown(
        f"Loaded `{len(nmrf_names)}` NMRFs from fixture `{run_id}`. "
        f"The chain builds `{valuation_request.spec_count}` valuation specs, "
        f"reconciles `{valuation_run.reconciliation.artifact_count}` returned artifacts, "
        f"and computes SES under `{policy.regime.value}`."
    )
)

## Implementation details / math verification - Method decisions and valuation artifacts

The remaining cells inspect method-selection inputs, valuation handoff records, and artifact loss tails.


## Method Selection Inputs

RFET determines the NMRF population. The selector then uses auditable evidence to choose a direct, stepwise, full-revaluation, or max-loss valuation contract. This fixture intentionally has one Type A direct NMRF and one Type B full-revaluation NMRF.

In [ ]:
method_rows: list[list[str]] = []
for name in nmrf_names:
    risk_factor = risk_factor_by_name[name]
    evidence = method_evidence[name]
    decision = decision_by_name[name]
    direct_diag = evidence.direct_robustness
    direct_value = "n/a" if direct_diag is None else f"{direct_diag.value:.3f} <= {direct_diag.threshold:.3f}"
    expected_method = fixture.expected_outputs["nmrf_methods"][name]
    method_rows.append(
        [
            name,
            classifications[name].value,
            risk_factor.risk_class.value,
            str(risk_factor.liquidity_horizon.value),
            direct_value,
            decision.method.value,
            decision.reason.value,
            str(decision.required_liquidity_horizon.value),
            "PASS" if decision.method.value == expected_method else "CHECK",
        ]
    )

display(
    markdown_table(
        [
            "Risk factor",
            "RFET status",
            "Class",
            "RF LH",
            "Direct diagnostic",
            "Selected method",
            "Reason",
            "Required LH",
            "Golden",
        ],
        method_rows,
    )
)

flags = [
    "direct available",
    "direct well-defined",
    "direct robust",
    "stepwise available",
    "full reval available",
    "nonlinear",
    "no pricing failures",
]
matrix = []
for name in nmrf_names:
    item = input_by_name[name]
    matrix.append(
        [
            item.direct_method_available,
            item.direct_shock_well_defined,
            item.direct_robust,
            item.stepwise_available,
            item.full_revaluation_available,
            item.nonlinear,
            method_evidence[name].pricing_failure_count == 0,
        ]
    )
matrix_values = np.asarray(matrix, dtype=int)

fig, ax = plt.subplots(figsize=(10.8, 3.2))
ax.imshow(matrix_values, cmap=ListedColormap(["#f2f2f2", "#4c78a8"]), vmin=0, vmax=1)
ax.set_xticks(np.arange(len(flags)), flags, rotation=25, ha="right")
ax.set_yticks(np.arange(len(nmrf_names)), nmrf_names)
for row in range(matrix_values.shape[0]):
    for col in range(matrix_values.shape[1]):
        ax.text(col, row, "Y" if matrix_values[row, col] else "N", ha="center", va="center")
ax.set_title("NMRF method-selection evidence flags")
fig.tight_layout()

## Valuation Spec Handoff and Reconciliation

The capital layer emits valuation specs and then reconciles returned upstream artifacts before any SES is consumed. Direct and full-revaluation pricing is not generated in this package.

In [ ]:
spec_rows: list[list[str]] = []
for spec in specs:
    if spec.direct_shock is not None:
        payload = f"{spec.direct_shock.shock_size:g} {spec.direct_shock.shock_unit} {spec.direct_shock.direction.value}"
    elif spec.full_revaluation is not None:
        payload = f"{spec.full_revaluation.scenario_set_id}; {len(spec.full_revaluation.market_state_ids)} states"
    elif spec.stepwise_grid is not None:
        payload = f"{spec.stepwise_grid.shock_count} shock points"
    elif spec.max_loss_fallback is not None:
        payload = f"{len(spec.max_loss_fallback.candidate_scenario_ids)} candidates"
    else:
        payload = spec.method.value
    spec_rows.append(
        [
            spec.risk_factor_name,
            spec.method.value,
            spec.risk_class.value,
            str(spec.required_liquidity_horizon.value),
            spec.stress_period.stress_period_id,
            payload,
        ]
    )

display(
    markdown_table(
        ["Risk factor", "Method", "Class", "Required LH", "Stress period", "Payload"],
        spec_rows,
    )
)

recon_rows: list[list[str]] = []
for item in valuation_run.reconciliation.items:
    recon_rows.append(
        [
            item.risk_factor_name,
            str(item.artifact_count),
            item.artifact_method.value if item.artifact_method is not None else "missing",
            str(item.artifact_loss_count),
            "PASS" if item.passed else "CHECK",
            ", ".join(item.errors) if item.errors else "none",
        ]
    )
display(
    markdown_table(
        ["Risk factor", "Artifacts", "Artifact method", "Loss count", "Status", "Errors"],
        recon_rows,
    )
)

check_names = ["method", "LH", "stress period", "scenario count", "scenario IDs", "source"]
check_matrix: list[list[int]] = []
check_text: list[list[str]] = []
for item in valuation_run.reconciliation.items:
    values = [
        item.method_matched,
        item.liquidity_horizon_matched,
        item.stress_period_matched,
        item.scenario_count_matched,
        item.scenario_ids_matched,
        item.source_present,
    ]
    row_codes: list[int] = []
    row_text: list[str] = []
    for value in values:
        if value is None:
            row_codes.append(2)
            row_text.append("n/a")
        elif value:
            row_codes.append(1)
            row_text.append("PASS")
        else:
            row_codes.append(0)
            row_text.append("FAIL")
    check_matrix.append(row_codes)
    check_text.append(row_text)

fig, ax = plt.subplots(figsize=(9.8, 3.4))
ax.imshow(
    np.asarray(check_matrix, dtype=int),
    cmap=ListedColormap(["#c44e52", "#59a14f", "#bab0ab"]),
    vmin=0,
    vmax=2,
)
ax.set_xticks(np.arange(len(check_names)), check_names, rotation=20, ha="right")
ax.set_yticks(
    np.arange(len(valuation_run.reconciliation.items)),
    [item.risk_factor_name for item in valuation_run.reconciliation.items],
)
for row in range(len(check_text)):
    for col in range(len(check_names)):
        ax.text(col, row, check_text[row][col], ha="center", va="center", fontsize=8)
ax.set_title("Valuation artifact reconciliation checks")
fig.tight_layout()

## Artifact Loss Tails and SES Extraction

NMRF artifacts use the package loss sign convention: positive values are losses. For direct and full-revaluation artifacts, SES is extracted from the expected-shortfall tail of the returned loss vector; max-loss fallback would use the maximum supplied loss.

In [ ]:
fig, axes = plt.subplots(1, len(nmrf_names), figsize=(12, 4.8), sharey=False)
if len(nmrf_names) == 1:
    axes = [axes]

for ax, name in zip(axes, nmrf_names, strict=True):
    artifact = artifact_by_name[name]
    losses = np.asarray(artifact.losses, dtype=float)
    tail_count, tail_cutoff, tail_mean = tail_summary(losses, policy.es_confidence_level)
    ses = ses_results_by_name[name].ses
    ax.hist(losses, bins=30, color="#4c78a8", alpha=0.82)
    ax.axvline(tail_cutoff, color="#111111", linestyle="--", linewidth=1.4, label="Tail cutoff")
    ax.axvline(ses, color="#c44e52", linewidth=1.8, label="SES")
    ax.set_title(f"{name}\n{artifact.method.value}; tail n={tail_count}")
    ax.set_xlabel("Stress loss, positive = loss")
    ax.set_ylabel("Scenario count")
    ax.text(
        0.03,
        0.94,
        f"tail mean={tail_mean:,.1f}\nSES={ses:,.1f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
    )
    ax.legend(frameon=False, loc="upper left")
fig.tight_layout()

## SES Aggregation and Golden Reconciliation

Under the prototype NPR 2.0 parameter set cited above, Type A NMRFs contribute through a zero-correlation sum of squares, while Type B NMRFs use the conservative partial-correlation term with policy `rho`. This notebook reconciles the computed total SES to the committed golden output used by capital assembly notebook 06.

In [ ]:
individual_rows: list[list[str]] = []
for name in nmrf_names:
    result = ses_results_by_name[name]
    individual_rows.append(
        [
            name,
            classifications[name].value,
            result.method.value,
            f"{result.ses:,.6f}",
            "NO" if not result.generated_by_prototype else "YES",
        ]
    )
display(
    markdown_table(
        ["Risk factor", "RFET status", "Artifact method", "SES", "Generated by prototype"],
        individual_rows,
    )
)

aggregation = nmrf_capital.aggregation
type_b_linear = aggregation.type_b_rho * aggregation.type_b_linear_sum**2
type_b_diversified = (1.0 - aggregation.type_b_rho) * aggregation.type_b_sum_of_squares
component_labels = ["Type A sum SES^2", "Type B rho x sum^2", "Type B residual sum^2"]
component_terms = np.asarray(
    [aggregation.type_a_sum_of_squares, type_b_linear, type_b_diversified],
    dtype=float,
)

expected_ses, ses_diff, ses_status = golden_check("total_ses", nmrf_capital.total_ses)
expected_reconciliation = fixture.expected_outputs["reconciliation"]
golden_rows = [
    ["Total SES", f"{nmrf_capital.total_ses:,.6f}", f"{expected_ses:,.6f}", f"{ses_diff:,.3e}", ses_status],
    ["Reconciliation passed", str(valuation_run.passed), str(expected_reconciliation["passed"]), "", "PASS" if valuation_run.passed == expected_reconciliation["passed"] else "CHECK"],
    ["Failed reconciliation items", str(valuation_run.reconciliation.failed_item_count), str(expected_reconciliation["failed_item_count"]), "", "PASS" if valuation_run.reconciliation.failed_item_count == expected_reconciliation["failed_item_count"] else "CHECK"],
    ["Returned artifacts", str(valuation_run.reconciliation.artifact_count), str(expected_reconciliation["artifact_count"]), "", "PASS" if valuation_run.reconciliation.artifact_count == expected_reconciliation["artifact_count"] else "CHECK"],
]
display(markdown_table(["Output", "Notebook", "Expected output", "Delta", "Status"], golden_rows))

fig, ax = plt.subplots(figsize=(9.4, 4.8))
ax.bar(component_labels, component_terms, color=["#4c78a8", "#f28e2b", "#59a14f"], alpha=0.88)
for index, term in enumerate(component_terms):
    ax.text(index, term, f"{term:,.0f}", ha="center", va="bottom")
ax.set_ylabel("SES squared contribution")
ax.set_title(f"SES aggregation terms; sqrt(total) = {aggregation.total_ses:,.2f}")
ax.tick_params(axis="x", rotation=18)
fig.tight_layout()

## See also

- [IMA package journey](../docs/PACKAGE_JOURNEY.md)
- [IMA dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [SBM package journey](../../frtb-sbm/docs/PACKAGE_JOURNEY.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [RRAO package journey](../../frtb-rrao/docs/PACKAGE_JOURNEY.md)
- [CVA package journey](../../frtb-cva/docs/PACKAGE_JOURNEY.md)
